# Bloque 3 — Agentes de IA
## Notebook 02: Agentes con herramientas (tool-use)

---

### El problema del notebook anterior

En el notebook 01, incrustramos todos los datos directamente en el texto de las tareas. Esto funciona para datasets pequeños, pero tiene dos limitaciones:

1. **Escala**: si hay 10,000 actores en vez de 30, no cabe en el contexto del LLM.
2. **Precisión**: el modelo puede "alucindar" al parafrasear los datos. Queremos que los números vengan de pandas, no de la imaginación del LLM.

### La solución: herramientas

Una **herramienta** es una función Python que el agente puede invocar cuando necesita información específica. El LLM decide **cuándo** llamarla y **con qué argumentos**.

```
┌─────────────────────────────────────────────────────────┐
│                        AGENTE                           │
│                                                         │
│  Pregunta: "¿Quién domina la categoría 'financial'?"    │
│                                                         │
│  Razona: "Necesito consultar la distribución...         │
│           voy a llamar top_actores_por_categoria()"     │
│                                                         │
│  Llama herramienta ──────────────────────────────────▶ │
│                    [top_actores_por_categoria('financial')]│
│  Recibe resultado ◀──────────────────────────────────── │
│                    ["stern: 12", "mango: 8", ...]       │
│                                                         │
│  Responde con datos reales (no alucinados)              │
└─────────────────────────────────────────────────────────┘
```

### El secreto del tool-use: el docstring

El LLM decide qué herramienta usar leyendo **su docstring**. Si el docstring es vago, el modelo elegirá mal. Si es preciso, elegirá bien.

```python
@tool
def mi_herramienta(arg: str) -> str:
    """Descripción precisa de CUÁNDO usar esta herramienta
    y QUÉ parámetros espera. El LLM lee esto para decidir."""
    ...
```

In [ ]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os
import json
import concurrent.futures
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

os.environ['OPENAI_API_KEY'] = 'NA'

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool


def kickoff(crew, timeout=600):
    """Ejecuta crew.kickoff() en un hilo separado para evitar conflictos
    con el event loop de Jupyter/nbconvert (problema de CrewAI 1.x)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result(timeout=timeout)


# ─── RUTAS ───────────────────────────────────────────────────────────────────
DATA = Path('..') / 'data_para_alumnos' / 'ContiLeaks' / 'data' / 'processed'

PROFILES_JSON    = DATA / 'actor_profiles.json'
CLASSIFIED_PARQ  = DATA / 'conti_sample_classified.parquet'
EMBEDDINGS_NPY   = DATA / 'message_embeddings.npy'

for p in [PROFILES_JSON, CLASSIFIED_PARQ, EMBEDDINGS_NPY]:
    assert p.exists(), f'No encontrado: {p}'

print('Rutas verificadas.')

In [ ]:
# ─── CARGAR Y PREPARAR DATOS ─────────────────────────────────────────────────
# Cargamos todos los datos UNA SOLA VEZ al inicio.
# Las herramientas accederán a estas variables globales sin recargar los ficheros.

with open(PROFILES_JSON) as f:
    PROFILES = json.load(f)

DF = pd.read_parquet(CLASSIFIED_PARQ)

# Cargamos los embeddings (24 MB, shape: 1500 × 4096).
# Cada fila es el vector semántico de un mensaje, en el mismo orden que DF.
EMBEDDINGS = np.load(EMBEDDINGS_NPY)

# Calculamos el centroide (vector promedio) de cada actor.
# El centroide representa el "estilo semántico" medio de ese actor.
# Ejemplo: si mango habla mucho de negociaciones, su centroide estará
# cerca de otros mensajes de negociación en el espacio vectorial.
ACTOR_CENTROIDS = {}
for actor, grupo in DF.groupby('username'):
    indices = grupo.index.tolist()
    centroide = EMBEDDINGS[indices].mean(axis=0)
    # Normalizamos el vector para que la similitud coseno sea más robusta.
    norma = np.linalg.norm(centroide)
    ACTOR_CENTROIDS[actor] = centroide / norma if norma > 0 else centroide

print(f'Datos cargados:')
print(f'  Actores con perfil    : {len(PROFILES)}')
print(f'  Mensajes clasificados : {len(DF)}')
print(f'  Shape embeddings      : {EMBEDDINGS.shape}')
print(f'  Centroides calculados : {len(ACTOR_CENTROIDS)}')

## Las tres herramientas

Vamos a definir tres herramientas con responsabilidades distintas:

| Herramienta | Cuándo usarla | Qué devuelve |
|---|---|---|
| `buscar_actor` | Cuando necesitas el perfil de un actor concreto | Rol, confianza, resumen |
| `top_actores_por_categoria` | Cuando necesitas saber quién domina una categoría | Lista de actores con conteos |
| `calcular_similitud` | Cuando necesitas comparar dos actores semánticamente | Similitud coseno 0-1 |

In [ ]:
# ─── HERRAMIENTA 1: buscar_actor ─────────────────────────────────────────────

@tool
def buscar_actor(nombre: str) -> str:
    """Busca y devuelve el perfil completo de un actor del grupo Conti por su nombre.
    Usar cuando se necesite información sobre el rol, nivel de confianza o actividad
    de un actor específico. El nombre debe estar en minúsculas (ej: 'stern', 'mango')."""

    # Normalizamos el nombre a minúsculas para evitar problemas de capitalización.
    nombre = nombre.lower().strip()

    if nombre not in PROFILES:
        # Si el actor no existe, devolvemos los disponibles para que el agente
        # pueda corregir el nombre en la siguiente iteración.
        disponibles = ', '.join(sorted(PROFILES.keys()))
        return f'Actor "{nombre}" no encontrado. Actores disponibles: {disponibles}'

    info = PROFILES[nombre]
    evidencias = '\n  '.join(info['evidence'][:3])  # máximo 3 ejemplos

    return (
        f'Actor: {nombre}\n'
        f'Rol: {info["role"]}\n'
        f'Confianza: {info["confidence"]}\n'
        f'Resumen: {info["summary"]}\n'
        f'Evidencias (hasta 3):\n  {evidencias}'
    )


# ─── HERRAMIENTA 2: top_actores_por_categoria ────────────────────────────────

@tool
def top_actores_por_categoria(categoria: str, n: int = 5) -> str:
    """Devuelve los N actores de Conti con más mensajes en una categoría dada.
    Categorías válidas: 'technical', 'comms', 'operational', 'financial',
    'organizational', 'unknown'. Usar cuando se quiera saber qué actores
    dominan un tipo de actividad concreto."""

    # Las categorías válidas son las que hay en el DataFrame.
    categorias_validas = sorted(DF['category'].unique())
    categoria = categoria.lower().strip()

    if categoria not in categorias_validas:
        return (
            f'Categoría "{categoria}" no válida. '
            f'Opciones: {categorias_validas}'
        )

    # Filtramos los mensajes de esa categoría y contamos por actor.
    conteos = (
        DF[DF['category'] == categoria]
        .groupby('username')
        .size()
        .sort_values(ascending=False)
        .head(n)
    )

    if conteos.empty:
        return f'No hay mensajes en la categoría "{categoria}".'

    lineas = [f'{actor}: {count} mensajes' for actor, count in conteos.items()]
    return f'Top {n} actores en categoría "{categoria}":\n' + '\n'.join(lineas)


# ─── HERRAMIENTA 3: calcular_similitud ───────────────────────────────────────

@tool
def calcular_similitud(actor1: str, actor2: str) -> str:
    """Calcula la similitud semántica (coseno) entre dos actores de Conti
    basándose en el estilo y contenido de sus mensajes (embeddings vectoriales).
    Un valor de 1.0 indica estilos idénticos; 0.0 indica estilos completamente distintos.
    Usar para comparar dos actores concretos o para encontrar quién es más similar a alguien."""

    actor1 = actor1.lower().strip()
    actor2 = actor2.lower().strip()

    actores_disponibles = list(ACTOR_CENTROIDS.keys())

    if actor1 not in ACTOR_CENTROIDS:
        return f'Actor "{actor1}" no encontrado. Disponibles: {actores_disponibles}'
    if actor2 not in ACTOR_CENTROIDS:
        return f'Actor "{actor2}" no encontrado. Disponibles: {actores_disponibles}'

    # cosine_similarity espera matrices 2D, por eso hacemos reshape a (1, N).
    v1 = ACTOR_CENTROIDS[actor1].reshape(1, -1)
    v2 = ACTOR_CENTROIDS[actor2].reshape(1, -1)
    sim = float(cosine_similarity(v1, v2)[0][0])

    if sim > 0.95:
        interpretacion = 'muy alta — estilos de comunicación muy similares'
    elif sim > 0.85:
        interpretacion = 'alta — patrones de comunicación parecidos'
    elif sim > 0.70:
        interpretacion = 'moderada — cierta similitud en el estilo'
    else:
        interpretacion = 'baja — estilos de comunicación distintos'

    return (
        f'Similitud entre "{actor1}" y "{actor2}": {sim:.4f}\n'
        f'Interpretación: {interpretacion}'
    )


print('Tres herramientas definidas: buscar_actor, top_actores_por_categoria, calcular_similitud')

## El agente con herramientas

Ahora creamos un único agente investigador que tiene acceso a las tres herramientas. El LLM decidirá cuándo y cuál usar basándose en la pregunta que le hacemos.

In [ ]:
# ─── CONFIGURACIÓN DEL LLM ───────────────────────────────────────────────────
llm = LLM(
    model='ollama/qwen2.5:14b',
    base_url='http://localhost:11434',
)

# ─── AGENTE CON HERRAMIENTAS ──────────────────────────────────────────────────
# La lista `tools` le dice al agente qué herramientas tiene disponibles.
# El backstory menciona explícitamente que debe usar las herramientas en vez
# de inventarse los datos — esto reduce las alucinaciones.

agente_investigador = Agent(
    role='Investigador analítico de grupos de ransomware',
    goal=(
        'Responder preguntas sobre el grupo Conti con datos precisos, '
        'usando las herramientas disponibles para consultar los datos reales.'
    ),
    backstory=(
        'Eres un investigador de CTI que trabaja con una base de datos de actores de Conti. '
        'Tienes acceso a tres herramientas de consulta: buscar_actor, top_actores_por_categoria '
        'y calcular_similitud. SIEMPRE usas estas herramientas para obtener datos precisos '
        'en vez de recordar información de memoria. Si necesitas el perfil de un actor, '
        'llamas a buscar_actor. Si necesitas rankings por actividad, llamas a '
        'top_actores_por_categoria. Si necesitas comparar dos actores, llamas a calcular_similitud. '
        'Respondes en español de forma concisa y factual.'
    ),
    tools=[buscar_actor, top_actores_por_categoria, calcular_similitud],
    llm=llm,
    verbose=True,
)

print('Agente con herramientas configurado.')

In [ ]:
# ─── FUNCIÓN AUXILIAR ─────────────────────────────────────────────────────────
# Creamos una función que lanza el crew con una pregunta y devuelve la respuesta.
# Así podemos reutilizarla en las celdas siguientes sin repetir código.

def preguntar(pregunta: str) -> str:
    """Lanza un crew con una pregunta y devuelve la respuesta del agente."""
    tarea = Task(
        description=pregunta,
        expected_output='Respuesta concisa y factual basada en los datos disponibles.',
        agent=agente_investigador,
    )
    crew = Crew(
        agents=[agente_investigador],
        tasks=[tarea],
        process=Process.sequential,
        verbose=True,
    )
    resultado = kickoff(crew)
    return resultado.raw

print('Función preguntar() lista.')

## Cinco preguntas de dificultad creciente

Vamos a observar cómo el agente decide qué herramienta usar para responder cada pregunta.

In [ ]:
# ─── PREGUNTA 1 (simple) — perfil de un actor específico ─────────────────────
# Herramienta esperada: buscar_actor('stern')
# El agente debería llamar directamente a buscar_actor con el nombre.

pregunta_1 = '¿Qué rol tiene el actor "stern" en el grupo Conti y cuál es su nivel de confianza?'

print(f'PREGUNTA 1: {pregunta_1}')
print('─' * 60)
respuesta_1 = preguntar(pregunta_1)
print('\nRESPUESTA:')
print(respuesta_1)

In [ ]:
# ─── PREGUNTA 2 (simple) — ranking por categoría ─────────────────────────────
# Herramienta esperada: top_actores_por_categoria('financial', 3)

pregunta_2 = '¿Qué tres actores de Conti tienen más mensajes de categoría financiera?'

print(f'PREGUNTA 2: {pregunta_2}')
print('─' * 60)
respuesta_2 = preguntar(pregunta_2)
print('\nRESPUESTA:')
print(respuesta_2)

In [ ]:
# ─── PREGUNTA 3 (media) — comparación semántica ──────────────────────────────
# Herramienta esperada: calcular_similitud('mango', 'stern')

pregunta_3 = '¿Qué tan similar es el estilo de comunicación de "mango" comparado con "stern"?'

print(f'PREGUNTA 3: {pregunta_3}')
print('─' * 60)
respuesta_3 = preguntar(pregunta_3)
print('\nRESPUESTA:')
print(respuesta_3)

In [ ]:
# ─── PREGUNTA 4 (media) — combinación de herramientas ────────────────────────
# Esta pregunta requiere usar DOS herramientas: primero buscar el perfil,
# luego comparar semánticamente. Observa si el agente las encadena.

pregunta_4 = (
    'Busca el perfil de "bentley" y luego compara su similitud semántica '
    'con "price". ¿Sus estilos de comunicación son consistentes con sus roles?'
)

print(f'PREGUNTA 4: {pregunta_4}')
print('─' * 60)
respuesta_4 = preguntar(pregunta_4)
print('\nRESPUESTA:')
print(respuesta_4)

In [ ]:
# ─── PREGUNTA 5 (compleja) — razonamiento sobre datos ────────────────────────
# Esta pregunta implica buscar quién domina la categoría técnica
# y luego buscar los perfiles de los top actores.
# El agente necesita encadenar múltiples llamadas a herramientas.

pregunta_5 = (
    'Identifica quién encabeza la actividad técnica en Conti y '
    'busca su perfil. ¿Es consistente su actividad técnica con el rol que se le asignó?'
)

print(f'PREGUNTA 5: {pregunta_5}')
print('─' * 60)
respuesta_5 = preguntar(pregunta_5)
print('\nRESPUESTA:')
print(respuesta_5)

## Análisis de fallos: las limitaciones de los modelos locales

Esta sección documenta los errores típicos que aparecen cuando usamos modelos locales (7B-14B) para tool-use, en comparación con GPT-4 o Claude.

### Fallo 1: Nombre de herramienta incorrecto

El modelo puede intentar llamar `buscar_perfil_actor` en vez de `buscar_actor`. Esto ocurre porque:
- El LLM recuerda vagamente el nombre de la función
- No tiene acceso al código fuente, solo al docstring

**Solución**: nombres de herramienta cortos y descriptivos. Evitar nombres ambiguos.

### Fallo 2: Argumentos mal tipados

El modelo puede llamar `top_actores_por_categoria('Financial', n='5')` con el n como string en vez de int. Los modelos más pequeños no siempre respetan los tipos de los parámetros.

**Solución**: validar y convertir tipos dentro de la herramienta (como hace nuestro `buscar_actor` con `nombre.lower().strip()`).

### Fallo 3: No llamar a la herramienta cuando debería

El modelo puede "recordar" el dato de su entrenamiento en vez de consultar la herramienta. Esto produce respuestas plausibles pero incorrectas para nuestros datos específicos.

**Solución**: incluir en el backstory la instrucción explícita de SIEMPRE usar las herramientas, y añadir en el `expected_output` que la respuesta debe citar el dato de la herramienta.

### Fallo 4: Bucle infinito de herramientas

Ocasionalmente el modelo llama a la misma herramienta repetidamente con argumentos ligeramente distintos. CrewAI tiene un límite de iteraciones (`max_iter`, por defecto 25) que previene esto.

### ¿Cuándo usar modelos más grandes?

| Tarea | 14B suficiente | Necesita 32B+ |
|---|---|---|
| Buscar un actor por nombre | ✅ | - |
| Ranking por categoría | ✅ | - |
| Comparar dos actores | ✅ | - |
| Encadenar 3+ herramientas | ⚠️ a veces falla | ✅ |
| Razonamiento sobre hallazgos contradictorios | ⚠️ | ✅ |
| Generación de hipótesis complejas | ❌ | ✅ |

Para un sistema de producción de CTI, se recomienda `qwen2.5:32b` o superior.

## Resumen del bloque 3

| Notebook | Concepto central |
|---|---|
| 00 | `Agent` + `Task` + `Crew`: los tres primitivos. Primer agente funcional. |
| 01 | Crew multi-agente secuencial con contexto encadenado. |
| 02 | Tool-use: el agente llama funciones Python para datos precisos. |

### ¿Qué sigue?

En el **bloque 4** aplicamos estas técnicas a conjuntos de datos reales de leaks de ransomware (Conti, BlackBasta, LockBit, Exploit.in) con análisis mucho más profundos. Las herramientas de este notebook son el puente entre el análisis clásico de pandas y un sistema de inteligencia aumentado por IA.